# Wild-Diff-ICMH — fine-tune Kgalagadi trên Colab

Chạy từng cell. Mặc định notebook chỉ smoke-test **20 optimizer step cho KGA:A01**; không tự đốt tài nguyên chạy cả 20 địa điểm.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, torch

REPO_URL = 'https://github.com/TranDuon/Wild-Diff-ICMH.git'
BRANCH = 'main'
REPO = Path('/content/Wild-Diff-ICMH')
if (REPO / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
assert torch.cuda.is_available(), 'Runtime > Change runtime type > chọn GPU'
gpu = torch.cuda.get_device_properties(0)
print(torch.__version__, torch.version.cuda, gpu.name, f'{gpu.total_memory/2**30:.1f} GiB')
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'compressai==1.2.8'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', 'src/recognize-anything'], check=True)
print('Cài xong. Nếu Colab yêu cầu restart runtime, restart rồi chạy lại từ đầu; các lệnh là idempotent.')


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/wild_diff_icmh')
DRIVE_IMAGES = DRIVE_ROOT / 'images'
LOCAL_IMAGES = Path('/content/data/wild_diff_icmh/images')
LOCAL_IMAGES.parent.mkdir(parents=True, exist_ok=True)
assert (DRIVE_IMAGES / 'snapshot_kgalagadi').is_dir(), f'Không thấy dữ liệu: {DRIVE_IMAGES}'
subprocess.run(['rsync', '-a', '--info=progress2', str(DRIVE_IMAGES) + '/', str(LOCAL_IMAGES) + '/'], check=True)
subprocess.run([sys.executable, 'tools/data/split_check.py', 'data/manifests/kgalagadi_site_split.jsonl', '--build-info', 'data/manifests/build_info.json'], check=True)


In [ ]:
from huggingface_hub import hf_hub_download
CKPT_ROOT = DRIVE_ROOT / 'checkpoints'
BPP_WEIGHT = 2
folder = f'CNscale1.0_1_1_{BPP_WEIGHT}_2_WTagGCM_bs16x1_lr0.00005_cfg7.0'
hf_hub_download(repo_id='Manojb/stable-diffusion-2-1-base', filename='v2-1_512-ema-pruned.ckpt', local_dir=CKPT_ROOT/'sd2p1')
hf_hub_download(repo_id='xinyu1205/recognize-anything-plus-model', filename='ram_plus_swin_large_14m.pth', local_dir=CKPT_ROOT/'ram')
hf_hub_download(repo_id='RuoyuFeng/Diff-ICMH', filename=f'difficmh_models/{folder}/model.ckpt', local_dir=CKPT_ROOT)
AUTHOR_CKPT = CKPT_ROOT / 'difficmh_models' / folder / 'model.ckpt'
repo_checkpoints = REPO / 'checkpoints'
if not repo_checkpoints.exists():
    repo_checkpoints.symlink_to(CKPT_ROOT, target_is_directory=True)
elif repo_checkpoints.resolve() != CKPT_ROOT.resolve():
    raise RuntimeError(f'{repo_checkpoints} đã tồn tại nhưng không trỏ tới {CKPT_ROOT}')
print(AUTHOR_CKPT)


In [ ]:
SITE = 'KGA:A01'
RAM_CKPT = CKPT_ROOT / 'ram' / 'ram_plus_swin_large_14m.pth'
TAGS_PATH = DRIVE_ROOT / 'tags' / f"{SITE.replace(':', '_')}.jsonl"
tag_command = [sys.executable, 'tools/precompute_ram_tags.py', '--data-root', str(LOCAL_IMAGES), '--checkpoint', str(RAM_CKPT), '--site-id', SITE, '--output', str(TAGS_PATH)]
subprocess.run(tag_command, cwd=REPO, check=True)


In [ ]:
RUN_DIR = DRIVE_ROOT / 'runs' / 'h1' / SITE.split(':')[-1]
env = os.environ.copy()
env.update(WILD_DATA_ROOT=str(LOCAL_IMAGES), KGA_SITE_ID=SITE, KGA_TAGS=str(TAGS_PATH), BPP_WEIGHT=str(BPP_WEIGHT), WILD_RUN_DIR=str(RUN_DIR), PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
command = [sys.executable, 'train.py', '--config', 'configs/train_kgalagadi_colab.yaml', '--init-checkpoint', str(AUTHOR_CKPT), 'lightning.trainer.max_steps=20', 'lightning.trainer.val_check_interval=10', 'lightning.trainer.check_val_every_n_epoch=1', 'lightning.trainer.limit_val_batches=2']
print(' '.join(command))
subprocess.run(command, cwd=REPO, env=env, check=True)


## Sau khi smoke test thành công

Xem VRAM/thời gian trong log. Sau đó bỏ bốn override smoke-test hoặc dùng `tools/train_kgalagadi_sites.py --max-sites 1`. Checkpoint nằm trên Drive và lần chạy sau tự resume từ `last.ckpt`. Xem `COLAB_TRAINING.md` cho H2, H3 và đánh giá.